# Day 4 · CUDA-Q 기반 Quantum-HPC 하이브리드 AI
**2026 서울퀀텀캠퍼스 양자컴퓨팅 활용과정 겨울캠프** · 12월 3일(목) · 연세대학교 박병규

오늘의 이야기: **"AI가 만든 신약 후보를, 양자 회로로 걸러 보자"**

| 시간 | 내용 |
|---|---|
| 10:00–11:00 | **Part 1** GPU와 QPU 연동 이해, 데이터를 주고받는 코드 작성 |
| 11:00–12:00 | **Part 2** 데이터를 양자 회로로 변환, 매개변수화 양자 회로(PQC) 만들기 |
| 13:30–15:00 | **Part 3** 학습, 힐베르트 공간의 확률 분포 보기, 기존 AI 모델과 비교 |
| 15:00–16:00 | **Part 4** 생성형 AI가 만든 후보를 양자 회로로 변환·평가 |
| 16:00–17:00 | **Part 5** 양자 계산 결과를 생성형 AI로 분석 |

> ✏️ = 직접 채우거나 바꿔 보는 셀 · 🤔 = 실행 전에 예측해 보는 질문

In [ ]:
import cudaq, time, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings; warnings.filterwarnings("ignore")

# GPU 시뮬레이터가 있으면 사용, 없으면 CPU
HAS_GPU = cudaq.has_target("nvidia")
cudaq.set_target("nvidia" if HAS_GPU else "qpp-cpu")
print("CUDA-Q:", cudaq.__version__.split("(")[0])
print("사용 중인 시뮬레이터:", cudaq.get_target().name, "(GPU)" if HAS_GPU else "(CPU)")

---
## Part 1 · GPU와 QPU 연동 이해, 데이터 통신 (10:00–11:00)

### 1-1. 하이브리드 컴퓨팅이란?
오늘날 양자컴퓨터(QPU)는 혼자 일하지 않습니다. **고전 컴퓨터(CPU·GPU)와 계속 대화**하면서 일합니다.

```
  ┌──────────── 고전 컴퓨터 (CPU/GPU) ────────────┐          ┌──── QPU ────┐
  │ 1. 데이터 준비 · 매개변수 θ 결정               │ ──θ, x──▶ │ 회로 실행    │
  │ 3. 결과로 손실 계산 → θ 업데이트 (최적화)      │ ◀─결과─── │ 측정 결과    │
  └────────────────────────────────────────────────┘          └─────────────┘
                     2 ↔ 3 을 수백~수천 번 반복
```
- **QPU**: 양자 회로를 실행하고 측정값을 돌려줌
- **CPU/GPU**: 데이터 처리, 최적화, 그리고 (지금처럼 실제 QPU가 작을 때는) **양자 회로 시뮬레이션**
- **CUDA-Q**: 같은 코드를 `set_target` 한 줄로 CPU 시뮬레이터, GPU 시뮬레이터, 실제 QPU 어디서든 실행

### 1-2. 실습 ① 큐비트가 늘면 왜 HPC가 필요할까?
큐비트 $n$개의 상태를 저장하려면 복소수 $2^n$개가 필요합니다. 30큐비트면 약 10억 개 → 16 GB.

In [ ]:
for n in [10, 20, 30, 40]:
    gb = (2 ** n) * 16 / 1e9          # complex128 = 16 bytes
    print(f"{n:2d} 큐비트: 상태 수 {2**n:>16,d}  메모리 {gb:>14,.3f} GB")

In [ ]:
@cudaq.kernel
def bench(n: int, layers: int):
    q = cudaq.qvector(n)
    for l in range(layers):
        for i in range(n):
            ry(0.3 * (i + 1), q[i])
        for i in range(n - 1):
            x.ctrl(q[i], q[i + 1])
    mz(q)

def time_it(target, sizes):
    cudaq.set_target(target)
    out = []
    for n in sizes:
        t = time.time(); cudaq.sample(bench, n, 5, shots_count=1000)
        out.append(time.time() - t)
        print(f"  [{target}] {n} qubits: {out[-1]:.2f} s")
    return out

# ✏️ CPU는 18~20큐비트부터 급격히 느려집니다. 너무 오래 걸리면 목록을 줄이세요.
cpu_sizes = [8, 12, 16, 18]
gpu_sizes = [8, 12, 16, 18, 22, 26]

t_cpu = time_it("qpp-cpu", cpu_sizes)
t_gpu = time_it("nvidia", gpu_sizes) if HAS_GPU else None

plt.figure(figsize=(5.5, 3.2))
plt.semilogy(cpu_sizes, t_cpu, "o-", label="CPU (qpp-cpu)")
if t_gpu: plt.semilogy(gpu_sizes, t_gpu, "o-", label="GPU (nvidia)")
plt.xlabel("qubits"); plt.ylabel("time (s, log)"); plt.legend(); plt.tight_layout(); plt.show()

cudaq.set_target("nvidia" if HAS_GPU else "qpp-cpu")   # 원래대로
if not HAS_GPU: print("※ 이 환경에는 GPU가 없어 CPU 결과만 표시했습니다.")

🤔 **질문** 그래프의 세로축은 로그 스케일입니다. 큐비트가 2개 늘 때마다 시간이 몇 배씩 늘어나나요? GPU는 어디쯤부터 차이가 커지나요?

### 1-3. 실습 ② 고전 ↔ 양자 데이터 주고받기
양자 커널에 **숫자(데이터·매개변수)를 보내고**, **기대값(숫자)을 돌려받는** 가장 작은 예입니다.

- 보내기: 커널 인자 `angles: list[float]`
- 받기: `cudaq.observe(...)` → $\langle Z \rangle$ 값 (−1 ~ +1)

In [ ]:
@cudaq.kernel
def tiny(angles: list[float]):
    q = cudaq.qvector(2)
    ry(angles[0], q[0])
    ry(angles[1], q[1])
    x.ctrl(q[0], q[1])

Z0 = cudaq.spin.z(0)

# ✏️ 각도를 바꿔 보세요 (라디안, 0 ~ 3.14)
angles = [___, ___]
value = cudaq.observe(tiny, Z0, angles).expectation()
print(f"보낸 값 {angles}  →  받은 값 <Z0> = {value:.4f}   (이론값 cos(θ0) = {np.cos(angles[0]):.4f})")

### 1-4. 한 번에 많이 보내기 (배치)
AI 학습에서는 데이터 수백 개를 계속 보내야 합니다. CUDA-Q는 **배열을 통째로 넘기면 한 번에 처리**합니다(broadcast). 내 컴퓨터 안의 시뮬레이터에서는 차이가 작지만, 원격 GPU 서버나 실제 QPU처럼 **호출할 때마다 통신 지연이 붙는 환경**에서는 묶어 보내는 것이 훨씬 중요해집니다.

In [ ]:
batch = np.random.default_rng(0).uniform(0, np.pi, size=(500, 2))

t = time.time()
one_by_one = [cudaq.observe(tiny, Z0, list(a)).expectation() for a in batch]
t1 = time.time() - t

t = time.time()
results = cudaq.observe(tiny, Z0, batch)          # 500개를 한 번에
batched = [r.expectation() for r in results]
t2 = time.time() - t

print(f"하나씩 500번: {t1:.2f} s   /   한 번에 500개: {t2:.2f} s   (결과 동일: {np.allclose(one_by_one, batched)})")

> **핵심 정리 (Part 1)**  
> - 하이브리드 = QPU는 회로 실행, CPU/GPU는 최적화·데이터 처리·시뮬레이션  
> - 큐비트 1개 추가 = 메모리·계산 2배 → 큰 시뮬레이션에는 GPU/HPC가 필수  
> - 데이터는 커널 인자로 보내고, 기대값·측정값으로 돌려받는다. 많이 보낼 때는 배치로

---
## Part 2 · 데이터를 양자 회로로, 매개변수화 양자 회로(PQC) (11:00–12:00)

### 2-1. 오늘의 데이터: "약이 될 수 있는 분자인가?"
분자를 4개의 숫자로 요약합니다.

| 특성 | 의미 | 약물 유사 기준 (Lipinski의 5의 법칙) |
|---|---|---|
| MW | 분자량 | ≤ 500 |
| logP | 기름에 녹는 정도 | ≤ 5 |
| HBD | 수소결합 주개 수 | ≤ 5 |
| HBA | 수소결합 받개 수 | ≤ 10 |

기준을 **2개 이상 어기면 "약물 유사하지 않음(0)"**, 그렇지 않으면 **"약물 유사(1)"** 로 표시합니다.  
(실습용으로 무작위 생성한 가상 분자 데이터입니다)

In [ ]:
rng = np.random.default_rng(7)
FEATURES = ["MW", "logP", "HBD", "HBA"]
LO = np.array([100, -3, 0, 0]); HI = np.array([800, 8, 9, 15])

def ro5_label(X):
    viol = (X[:, 0] > 500).astype(int) + (X[:, 1] > 5) + (X[:, 2] > 5) + (X[:, 3] > 10)
    return (viol <= 1).astype(int)

def make_molecules(n):
    X = np.c_[rng.uniform(100, 800, n), rng.uniform(-3, 8, n),
              rng.integers(0, 10, n), rng.integers(0, 16, n)].astype(float)
    return X, ro5_label(X)

X_raw, y = make_molecules(300)
X_tr_raw, y_tr, X_te_raw, y_te = X_raw[:200], y[:200], X_raw[200:], y[200:]
print(f"학습 {len(y_tr)}개 / 테스트 {len(y_te)}개,  약물 유사 비율 {y.mean():.0%}")
print("예시 5개:")
for r, l in zip(X_raw[:5], y[:5]):
    print("  ", {f: round(float(v), 1) for f, v in zip(FEATURES, r)}, "→", l)

### 2-2. 숫자를 각도로: 각도 인코딩 (angle encoding)
큐비트는 숫자를 그대로 넣을 수 없습니다. 각 특성을 $0 \sim \pi$ 사이 각도로 바꾼 뒤, **큐비트 하나당 특성 하나**씩 $R_y(\text{각도})$ 회전으로 넣습니다.

$$x \;\rightarrow\; \theta = \pi\cdot\frac{x - x_{\min}}{x_{\max}-x_{\min}} \;\rightarrow\; R_y(\theta)\lvert0\rangle = \cos\tfrac{\theta}{2}\lvert0\rangle + \sin\tfrac{\theta}{2}\lvert1\rangle$$

특성 4개 → 큐비트 4개 → $2^4 = 16$차원 **힐베르트 공간**.

In [ ]:
def encode(X):
    return (X - LO) / (HI - LO) * np.pi

X_tr, X_te = encode(X_tr_raw), encode(X_te_raw)

@cudaq.kernel
def encode_only(f: list[float]):
    q = cudaq.qvector(4)
    for i in range(4):
        ry(f[i], q[i])

i = 0
print("분자", {f: round(float(v), 1) for f, v in zip(FEATURES, X_tr_raw[i])}, "→ 각도", np.round(X_tr[i], 2))
print(cudaq.draw(encode_only, list(X_tr[i])))

### 2-3. 매개변수화 양자 회로 (PQC) = 양자 신경망의 층
신경망에 가중치가 있듯이, PQC에는 **학습 가능한 회전 각도 θ**가 있습니다.

한 층(layer)의 구성:
1. 각 큐비트에 $R_y(\theta)$, $R_z(\theta)$ (학습되는 부분)
2. CNOT으로 큐비트들을 고리 모양으로 **얽기** (특성끼리 정보를 섞음)
3. 데이터를 한 번 더 넣기 (**data re-uploading**: 표현력을 높이는 기법)

마지막으로 각 큐비트의 $\langle Z_i\rangle$ 4개를 읽고, 고전 가중치 $w$로 묶어 확률을 냅니다.
$$p(\text{약물 유사}) = \sigma\Big(\textstyle\sum_i w_i\langle Z_i\rangle + b\Big)$$
**양자 층 + 고전 출력층** → 이것이 하이브리드 AI 모델입니다.

In [ ]:
@cudaq.kernel
def pqc(f: list[float], theta: list[float], layers: int):
    q = cudaq.qvector(4)
    for i in range(4):                     # 데이터 인코딩
        ry(f[i], q[i])
    k = 0
    for l in range(layers):
        for i in range(4):                 # 학습 가능한 회전
            ry(theta[k], q[i]); k += 1
            rz(theta[k], q[i]); k += 1
        for i in range(4):                 # 얽힘 (고리)
            x.ctrl(q[i], q[(i + 1) % 4])
        for i in range(4):                 # 데이터 재입력
            ry(f[i], q[i])

def n_theta(layers): return 8 * layers

print(cudaq.draw(pqc, list(X_tr[0]), list(np.zeros(n_theta(1))), 1))

In [ ]:
Z = [cudaq.spin.z(i) for i in range(4)]
H_all = Z[0] + Z[1] + Z[2] + Z[3]
CALLS = {"circuits": 0}                    # 양자 회로 실행 횟수 세기

def z_features(F, theta, layers):
    # 데이터 여러 개를 한 번에 보내고, 각 큐비트의 <Z_i>를 받아오기
    F = np.asarray(F)
    res = cudaq.observe(pqc, H_all, F, np.tile(theta, (len(F), 1)), np.full(len(F), layers))
    CALLS["circuits"] += len(F)
    return np.array([[r.expectation(z) for z in Z] for r in res])

def predict_proba(F, params, layers):
    th, w, b = params[:n_theta(layers)], params[n_theta(layers):-1], params[-1]
    return 1 / (1 + np.exp(-(z_features(F, th, layers) @ w + b)))

# 학습 전(무작위 θ) 예측
p0 = np.r_[rng.normal(0, 0.3, n_theta(1)), np.ones(4), 0.0]
print("학습 전 예측 확률(5개):", np.round(predict_proba(X_tr[:5], p0, 1), 3), " 정답:", y_tr[:5])

> **핵심 정리 (Part 2)**  
> - 특성 → 각도 → $R_y$ 회전 (큐비트 1개당 특성 1개)  
> - PQC = 학습되는 회전 + 얽힘 + (데이터 재입력)의 반복  
> - 양자 층의 출력 $\langle Z_i\rangle$를 고전 층이 받아 최종 판단 → 하이브리드 모델

---
## Part 3 · 학습, 힐베르트 공간의 확률 분포, 기존 AI와 비교 (13:30–15:00)

### 3-1. 하이브리드 학습 루프
손실 함수(교차 엔트로피)를 최소화하는 θ, w, b를 고전 최적화기(COBYLA)로 찾습니다. 매 반복마다 **고전 → 양자 → 고전** 통신이 200번(학습 데이터 수)씩 일어납니다.

In [ ]:
def train(layers, maxiter=400, seed=1):
    r = np.random.default_rng(seed)
    params0 = np.r_[r.normal(0, 0.3, n_theta(layers)), np.ones(4), 0.0]
    history = []
    def loss(params):
        p = np.clip(predict_proba(X_tr, params, layers), 1e-6, 1 - 1e-6)
        L = -np.mean(y_tr * np.log(p) + (1 - y_tr) * np.log(1 - p))
        history.append(L); return L
    t = time.time()
    res = minimize(loss, params0, method="COBYLA", options={"maxiter": maxiter})
    return res.x, history, time.time() - t

def accuracy(F, labels, params, layers):
    return ((predict_proba(F, params, layers) > 0.5) == labels).mean()

# ✏️ 층 수를 정하세요 (1 ~ 3)
LAYERS = ___
CALLS["circuits"] = 0
params_q, hist, sec = train(LAYERS)

print(f"학습 시간 {sec:.1f} s,  양자 회로 실행 {CALLS['circuits']:,}회")
print(f"정확도  학습 {accuracy(X_tr, y_tr, params_q, LAYERS):.0%}   테스트 {accuracy(X_te, y_te, params_q, LAYERS):.0%}")
plt.figure(figsize=(5, 2.6)); plt.plot(hist); plt.xlabel("iteration"); plt.ylabel("loss")
plt.title(f"hybrid training ({LAYERS} layers)"); plt.tight_layout(); plt.show()

🤔 **질문** 실제 QPU에서 이 학습을 한다면, 위의 "양자 회로 실행 횟수" × shots(예: 1,000)만큼 측정해야 합니다. 시간과 비용은 어떨까요? → 그래서 학습은 **GPU 시뮬레이터**로, 최종 검증만 **QPU**로 하는 방식이 현실적입니다.

### 3-2. 16차원 힐베르트 공간 들여다보기
학습된 회로에 분자를 넣고 4큐비트를 측정하면, 16개 기저 상태(`0000`~`1111`)에 대한 **확률 분포**가 나옵니다. 약물 유사 분자와 그렇지 않은 분자의 분포가 다르게 학습되었을까요?

In [ ]:
def hilbert_dist(f, params, layers, shots=2000):
    r = cudaq.sample(pqc, list(f), list(params[:n_theta(layers)]), layers, shots_count=shots)
    p = np.zeros(16)
    for bits, c in r.items():
        p[int(bits[::-1], 2)] += c / shots
    return p

idx1 = np.where(y_te == 1)[0][:20]; idx0 = np.where(y_te == 0)[0][:20]
d1 = np.mean([hilbert_dist(X_te[i], params_q, LAYERS) for i in idx1], axis=0)
d0 = np.mean([hilbert_dist(X_te[i], params_q, LAYERS) for i in idx0], axis=0)

labels = [format(k, "04b")[::-1] for k in range(16)]
xs = np.arange(16)
plt.figure(figsize=(9, 3))
plt.bar(xs - 0.2, d1, 0.4, label="drug-like (1)", color="#4C72B0")
plt.bar(xs + 0.2, d0, 0.4, label="not drug-like (0)", color="#DD8452")
plt.xticks(xs, labels, rotation=90); plt.xlabel("basis state q0q1q2q3"); plt.ylabel("avg probability")
plt.legend(); plt.title("Learned distributions in 16-dim Hilbert space"); plt.tight_layout(); plt.show()

### 3-3. 기존 AI 모델과 비교
같은 데이터로 **로지스틱 회귀**(가장 기본적인 고전 분류기)를 학습해 비교합니다.

In [ ]:
def train_logreg(F, labels, lr=0.1, steps=5000):
    A = np.c_[F, np.ones(len(F))]; w = np.zeros(A.shape[1])
    for _ in range(steps):
        p = 1 / (1 + np.exp(-A @ w)); w -= lr * A.T @ (p - labels) / len(labels)
    return w

def logreg_proba(F, w): return 1 / (1 + np.exp(-np.c_[F, np.ones(len(F))] @ w))

w_lr = train_logreg(X_tr, y_tr)
acc_lr = ((logreg_proba(X_te, w_lr) > 0.5) == y_te).mean()

n_params_q = n_theta(LAYERS) + 5
print(f"{'모델':<22}{'매개변수 수':>10}{'테스트 정확도':>14}")
print(f"{'로지스틱 회귀 (고전)':<22}{5:>10}{acc_lr:>14.0%}")
print(f"{f'하이브리드 PQC ({LAYERS}층)':<22}{n_params_q:>10}{accuracy(X_te, y_te, params_q, LAYERS):>14.0%}")

### 3-4. 탐색: 층 수에 따라 어떻게 달라질까?
층이 많을수록 표현력은 커지지만, 데이터가 적으면 **과적합**이 생길 수 있습니다.

In [ ]:
# ✏️ 시간이 부족하면 [1, 2]로 줄이세요
layer_list = [1, 2, 3]
rows = []
for L in layer_list:
    prm, _, s = train(L, maxiter=300)
    rows.append((L, accuracy(X_tr, y_tr, prm, L), accuracy(X_te, y_te, prm, L), s))
    print(f"{L}층: 학습 {rows[-1][1]:.0%}  테스트 {rows[-1][2]:.0%}  ({s:.1f} s)")

plt.figure(figsize=(4.5, 2.8))
plt.plot(layer_list, [r[1] for r in rows], "o-", label="train")
plt.plot(layer_list, [r[2] for r in rows], "o-", label="test")
plt.axhline(acc_lr, color="gray", ls="--", label="logistic regression")
plt.xlabel("layers"); plt.ylabel("accuracy"); plt.legend(); plt.tight_layout(); plt.show()

🤔 **토론** 양자 모델이 고전 모델보다 확실히 좋았나요?  
→ 이 정도 크기의 문제에서는 **비슷한 수준**인 것이 정상입니다. 지금의 양자 머신러닝 연구는 "어떤 데이터와 구조에서 양자가 유리한가"를 찾는 단계입니다. **정직한 비교**가 가장 중요한 연구 습관입니다.

> **핵심 정리 (Part 3)**  
> - 학습 = 고전 최적화기가 양자 회로를 수천~수만 번 호출하는 루프  
> - 데이터는 16차원 힐베르트 공간의 확률 분포로 표현된다  
> - 층 수·데이터 수에 따라 성능이 달라지며, 고전 모델과의 공정한 비교가 필수

---
## Part 4 · 생성형 AI가 만든 후보를 양자 회로로 (15:00–16:00)

### 4-1. 시나리오
신약 개발에서는 **생성형 AI가 새로운 분자 후보를 대량으로 제안**하고, 다른 모델들이 이를 걸러냅니다. 오늘은
1. 생성형 AI가 후보 분자 8개를 제안했다고 하고 (아래 JSON)
2. 우리가 학습한 **하이브리드 양자 모델**로 "약물 유사도"를 평가해 봅니다.

> 아래 후보는 실습용으로 만든 **가상의 분자**입니다. 4-3에서 직접 생성형 AI에게 후보를 받아 바꿔 넣을 수 있습니다.

In [ ]:
candidates_json = '''
[
 {"id": "CAND-01", "MW": 320, "logP": 2.1,  "HBD": 2, "HBA": 5},
 {"id": "CAND-02", "MW": 610, "logP": 3.0,  "HBD": 3, "HBA": 8},
 {"id": "CAND-03", "MW": 720, "logP": 6.2,  "HBD": 4, "HBA": 11},
 {"id": "CAND-04", "MW": 450, "logP": 5.6,  "HBD": 1, "HBA": 6},
 {"id": "CAND-05", "MW": 540, "logP": 5.8,  "HBD": 2, "HBA": 7},
 {"id": "CAND-06", "MW": 280, "logP": -1.2, "HBD": 6, "HBA": 9},
 {"id": "CAND-07", "MW": 680, "logP": 1.5,  "HBD": 7, "HBA": 13},
 {"id": "CAND-08", "MW": 390, "logP": 4.4,  "HBD": 3, "HBA": 7}
]
'''
cands = json.loads(candidates_json)
C_raw = np.array([[c[f] for f in FEATURES] for c in cands], dtype=float)
C_enc = encode(np.clip(C_raw, LO, HI))
print(f"후보 {len(cands)}개 불러옴")

In [ ]:
p_quantum = predict_proba(C_enc, params_q, LAYERS)
p_classic = logreg_proba(C_enc, w_lr)
truth = ro5_label(C_raw)

print(f"{'ID':<9}{'MW':>5}{'logP':>6}{'HBD':>4}{'HBA':>4} | {'양자 점수':>8}{'고전 점수':>8}  {'규칙 정답':>6}  {'양자 판정':>6}")
for c, pq, pc, t in zip(cands, p_quantum, p_classic, truth):
    ok = "✓" if (pq > 0.5) == t else "✗"
    print(f"{c['id']:<9}{c['MW']:>5}{c['logP']:>6}{c['HBD']:>4}{c['HBA']:>4} | {pq:>8.2f}{pc:>8.2f}  {t:>6}  {ok:>6}")

### 4-2. 후보 하나가 회로가 되는 모습

In [ ]:
# ✏️ 보고 싶은 후보 번호 (0 ~ 7)
k = 0
print(cands[k]["id"], "→ 각도", np.round(C_enc[k], 2))
print(cudaq.draw(pqc, list(C_enc[k]), list(params_q[:n_theta(LAYERS)]), 1))   # 1층만 그림

dist = hilbert_dist(C_enc[k], params_q, LAYERS)
plt.figure(figsize=(8, 2.6)); plt.bar(range(16), dist)
plt.xticks(range(16), labels, rotation=90); plt.ylabel("probability")
plt.title(f"{cands[k]['id']} in Hilbert space"); plt.tight_layout(); plt.show()

### 4-3. 실습 ③ 생성형 AI에게 직접 후보 받기
Claude, ChatGPT 같은 생성형 AI에게 아래 문장을 그대로 붙여 넣고, 돌아온 JSON을 다음 셀의 `my_json`에 붙여 넣으세요.

In [ ]:
gen_prompt = '''신약 후보 분자 6개를 제안해 줘. 각 분자는 MW(분자량, 100~800), logP(-3~8),
HBD(수소결합 주개 수, 0~9 정수), HBA(수소결합 받개 수, 0~15 정수)만 가진다.
약물 유사한 후보와 그렇지 않은 후보를 섞어 줘.
설명 없이 다음 형식의 JSON 배열만 출력해:
[{"id": "GEN-01", "MW": 0, "logP": 0, "HBD": 0, "HBA": 0}]'''
print(gen_prompt)

In [ ]:
# ✏️ 생성형 AI가 준 JSON을 붙여 넣으세요 (비워 두면 건너뜀)
my_json = ''

if my_json.strip():
    gen = json.loads(my_json)
    G_raw = np.array([[g[f] for f in FEATURES] for g in gen], dtype=float)
    G_enc = encode(np.clip(G_raw, LO, HI))
    for g, pq, t in zip(gen, predict_proba(G_enc, params_q, LAYERS), ro5_label(G_raw)):
        print(f"{g['id']:<8} 양자 점수 {pq:.2f}  규칙 정답 {t}")
else:
    print("JSON이 비어 있어 건너뜁니다.")

> **핵심 정리 (Part 4)**  
> - 생성형 AI(후보 생성) → 양자 모델(평가)로 이어지는 **AI–양자 파이프라인**  
> - 후보 → 정규화 → 각도 → 회로, 학습 데이터와 **똑같은 방식**으로 변환해야 한다  
> - 모델 점수는 규칙(정답)과 다를 수 있다 → 어디서 틀리는지가 다음 분석의 출발점

---
## Part 5 · 양자 계산 결과를 생성형 AI로 분석 (16:00–17:00)

양자 모델이 낸 숫자들을 사람이 이해하기 쉬운 말로 정리하는 데 생성형 AI를 씁니다. 결과를 **표 형태 텍스트**로 만들어 질문과 함께 보냅니다.

In [ ]:
rows_txt = "\n".join(
    f"{c['id']}: MW={c['MW']}, logP={c['logP']}, HBD={c['HBD']}, HBA={c['HBA']}, "
    f"quantum_score={pq:.2f}, classical_score={pc:.2f}, rule_label={t}"
    for c, pq, pc, t in zip(cands, p_quantum, p_classic, truth))

analysis_prompt = f'''다음은 4큐비트 하이브리드 양자 분류기({LAYERS}층 PQC)와 로지스틱 회귀가
신약 후보 분자의 약물 유사도(Lipinski 5의 법칙)를 예측한 결과야.
score는 약물 유사할 확률이고, rule_label은 규칙으로 계산한 정답(1=약물 유사)이야.
모델 테스트 정확도: 양자 {accuracy(X_te, y_te, params_q, LAYERS):.0%}, 고전 {acc_lr:.0%}

{rows_txt}

1) 다음 실험 단계로 보낼 후보 3개를 추천하고 이유를 말해 줘.
2) 양자 모델이 규칙과 다르게 판단한 후보가 있다면 어떤 특성 때문인지 추정해 줘.
3) 이 결과만으로 내릴 수 없는 결론(한계)을 2가지 적어 줘.
입문자도 이해할 수 있게 한국어로 짧게 답해 줘.'''
print(analysis_prompt)

### 5-1. 생성형 AI에게 보내기
- **방법 A (누구나):** 위에 출력된 내용을 복사해서 Claude나 ChatGPT 웹 화면에 붙여 넣습니다.
- **방법 B (API 키가 있을 때):** 아래 셀에서 코드로 바로 보냅니다. `ANTHROPIC_API_KEY` 환경변수와 `anthropic` 패키지가 필요합니다.

In [ ]:
import os
MODEL = "claude-sonnet-5-5"      # 사용할 모델 이름

if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic
    client = anthropic.Anthropic()
    msg = client.messages.create(model=MODEL, max_tokens=1000,
                                 messages=[{"role": "user", "content": analysis_prompt}])
    print(msg.content[0].text)
else:
    print("API 키가 없어 건너뜁니다 → 방법 A로 진행하세요.")

### 5-2. 토론: 생성형 AI의 분석을 믿어도 될까?
생성형 AI의 답을 받은 뒤 함께 확인해 봅시다.
- AI가 추천한 후보가 **규칙(정답)** 과 맞나요?
- 양자 점수의 차이를 **과장**해서 해석하지 않았나요? (예: 0.55와 0.60의 차이)
- "양자라서 더 정확하다" 같은 **근거 없는 주장**이 들어 있지 않나요?

> 생성형 AI는 **해석과 정리**를 돕는 도구이고, 최종 판단의 근거는 **데이터와 검증**입니다.

---
## 오늘의 정리
1. 하이브리드 컴퓨팅 = QPU(회로 실행) + CPU/GPU(최적화·시뮬레이션), CUDA-Q로 한 코드
2. 데이터 → 각도 인코딩 → PQC → $\langle Z\rangle$ → 고전 출력층 = 하이브리드 AI 모델
3. 학습은 양자 회로를 수만 번 호출 → GPU 시뮬레이터가 핵심 인프라
4. 고전 모델과 공정하게 비교하고, 생성형 AI의 해석은 반드시 검증